# Isolation Forest Evaluation

Evaluates each of the four fitted Isolation Forest models against three test sets separately:

| Test Set | Type | Source |
|----------|------|---------|
| CAPTCHA | True positives | `data/hammer_captchas/` |
| CIFAR-10 | Easy negatives | `data/cifar10/` |
| SVHN | Hard negatives | `data/svhn/` |

Metrics reported per dataset: **ROC-AUC** and **FPR at 95% TPR**.

In [1]:
import os
import random
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from dotenv import find_dotenv, load_dotenv
from sklearn.metrics import roc_auc_score, roc_curve
from torch.utils.data import DataLoader, Subset
from torchvision import transforms

load_dotenv(find_dotenv(), override=True)

PROJECT_ROOT = Path(os.environ["PROJECT_ROOT_DIR"])
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from src.datasets.cifar10dataset import CIFAR10Dataset
from src.datasets.kaggledataset import KaggleDataset
from src.datasets.svhndataset import SVHNDataset
from src.models.isolation_forest.feature_extractor import CaptchaFeatureExtractor

## Configuration

In [2]:
SEED        = 99   # Different seed from training to ensure unseen CAPTCHA samples
NUM_SAMPLES = 10_000
BATCH_SIZE  = 256

MODEL_NAMES = [
    "crnn_base",
    "crnn_finetuned",
    "convtrans_base",
    "convtrans_finetuned",
]

WEIGHTS_DIR = PROJECT_ROOT / "weights" / "isolation_forest"

## Helper Functions

In [3]:
def collate_images_only(batch):
    images = torch.stack([item[0] for item in batch])
    return (images,)


def make_dataloader(dataset, num_samples=NUM_SAMPLES, seed=SEED):
    """Subsample a dataset and wrap it in a DataLoader."""
    random.seed(seed)
    indices = random.sample(range(len(dataset)), min(num_samples, len(dataset)))
    subset = Subset(dataset, indices)
    return DataLoader(
        subset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        collate_fn=collate_images_only,
    )


def fpr_at_tpr(y_true, scores, tpr_threshold=0.95):
    """Return the FPR at the threshold where TPR >= tpr_threshold."""
    fpr, tpr, _ = roc_curve(y_true, scores)
    idx = np.searchsorted(tpr, tpr_threshold)
    idx = min(idx, len(fpr) - 1)
    return fpr[idx]

## Load Datasets

In [4]:
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((40, 150)),
    transforms.ToTensor(),
])

captcha_dataset = KaggleDataset(
    root_dir=str(PROJECT_ROOT / "data" / "hammer_captchas"),
    transform=transform,
    preload=False,
)
cifar10_dataset = CIFAR10Dataset(
    root_dir=str(PROJECT_ROOT / "data" / "cifar10"),
    transform=transform,
)
svhn_dataset = SVHNDataset(
    root_dir=str(PROJECT_ROOT / "data" / "svhn"),
    transform=transform,
)

print(f"CAPTCHA images : {len(captcha_dataset):>8,}")
print(f"CIFAR-10 images: {len(cifar10_dataset):>8,}")
print(f"SVHN images    : {len(svhn_dataset):>8,}")

captcha_loader = make_dataloader(captcha_dataset)
cifar10_loader = make_dataloader(cifar10_dataset)
svhn_loader    = make_dataloader(svhn_dataset)

print(f"\nEvaluating with {NUM_SAMPLES:,} samples per dataset")

CAPTCHA images : 1,365,874
CIFAR-10 images:   10,000
SVHN images    :   26,032

Evaluating with 10,000 samples per dataset


## Evaluation

For each model:
1. Load the fitted Isolation Forest from disk
2. Extract features from each test set using the frozen CNN backbone
3. Score with `decision_function()` — higher = more CAPTCHA-like
4. Compute ROC-AUC and FPR@95TPR per dataset

In [5]:
results = []

test_sets = {
    "CAPTCHA":  (captcha_loader, 1),
    "CIFAR-10": (cifar10_loader, 0),
    "SVHN":     (svhn_loader,    0),
}

for model_name in MODEL_NAMES:
    print(f"\n{'=' * 60}")
    print(f"  {model_name}")
    print(f"{'=' * 60}")

    # Load fitted Isolation Forest
    clf_path = WEIGHTS_DIR / f"{model_name}.joblib"
    if not clf_path.exists():
        print(f"  WARNING: {clf_path} not found — skipping")
        continue
    clf = joblib.load(clf_path)

    # Load feature extractor
    extractor = CaptchaFeatureExtractor(model_name)

    # Collect features + labels across all test sets
    all_features = []
    all_labels   = []
    set_features = {}

    for set_name, (loader, label) in test_sets.items():
        print(f"  Extracting features: {set_name}...")
        feats = extractor.extract(loader)
        set_features[set_name] = (feats, label)
        all_features.append(feats)
        all_labels.extend([label] * len(feats))

    # Evaluate per dataset
    for set_name, (feats, label) in set_features.items():
        # Build binary labels: 1=CAPTCHA vs this set
        captcha_feats, _ = set_features["CAPTCHA"]

        if set_name == "CAPTCHA":
            continue  # Need a negative class to compute ROC-AUC

        y_true  = np.array([1] * len(captcha_feats) + [0] * len(feats))
        scores  = clf.decision_function(
            np.concatenate([captcha_feats, feats], axis=0)
        )

        auc     = roc_auc_score(y_true, scores)
        fpr_95  = fpr_at_tpr(y_true, scores, tpr_threshold=0.95)

        print(f"    vs {set_name:8s} — ROC-AUC: {auc:.4f}  |  FPR@95TPR: {fpr_95:.4f}")
        results.append({
            "Model":      model_name,
            "Negatives":  set_name,
            "ROC-AUC":    round(auc, 4),
            "FPR@95TPR":  round(fpr_95, 4),
        })

print("\nEvaluation complete.")


  crnn_base
Loading Graf-J/captcha-crnn-base on cuda...


W0309 20:08:54.542000 14024 .venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/38 [00:00<?, ?it/s]

  Extracting features: CAPTCHA...


Extracting [crnn_base]: 100%|██████████| 40/40 [00:08<00:00,  4.85it/s]


...

  Extracting features: CIFAR-10...


Extracting [convtrans_finetuned]: 100%|██████████| 40/40 [00:05<00:00,  7.61it/s]


  Extracting features: SVHN...


Extracting [convtrans_finetuned]: 100%|██████████| 40/40 [00:05<00:00,  7.85it/s]


    vs CIFAR-10 — ROC-AUC: 0.9939  |  FPR@95TPR: 0.0041
    vs SVHN     — ROC-AUC: 0.9917  |  FPR@95TPR: 0.0086

Evaluation complete.


## Results Summary

In [6]:
df = pd.DataFrame(results)
pivot = df.pivot(index="Model", columns="Negatives", values=["ROC-AUC", "FPR@95TPR"])
pivot.columns = [f"{metric} vs {neg}" for metric, neg in pivot.columns]
pivot = pivot.sort_index()

print("\n=== Isolation Forest Evaluation Results ===")
print(pivot.to_string())
pivot


=== Isolation Forest Evaluation Results ===
                     ROC-AUC vs CIFAR-10  ROC-AUC vs SVHN  FPR@95TPR vs CIFAR-10  FPR@95TPR vs SVHN
Model                                                                                              
convtrans_base                    0.9951           0.9940                 0.0062             0.0056
convtrans_finetuned               0.9939           0.9917                 0.0041             0.0086
crnn_base                         0.9834           0.9883                 0.0217             0.0110
crnn_finetuned                    0.9792           0.9858                 0.0172             0.0085


,ROC-AUC vs CIFAR-10,ROC-AUC vs SVHN,FPR@95TPR vs CIFAR-10,FPR@95TPR vs SVHN
Model,,,,
convtrans_base,0.9951,0.9940,0.0062,0.0056
convtrans_finetuned,0.9939,0.9917,0.0041,0.0086
crnn_base,0.9834,0.9883,0.0217,0.0110
crnn_finetuned,0.9792,0.9858,0.0172,0.0085
